# 2. 餐廳與情境抽取（台中美食版）

改自組員的台北版 `經驗抽取.ipynb`：從 `1_youtube爬蟲` 的字幕抽出**餐廳**、**情境片段**（DESIGN.md §9.1 標籤表）與**價格**，經程式查核與人工審核後，輸出原型用的 `restaurants.json`。

| 步驟 | 內容 | 需要金鑰 |
|---|---|---|
| 1 | 設定 | |
| 2 | 清理字幕：去掉重複句、標出非中文字幕 | |
| 3 | Gemini 抽取餐廳＋情境片段＋價格（先試跑少量） | `GEMINI_API_KEY` |
| 4 | 攤平並自動查核：引文逐字、標籤在表內、數字有根據、店名可交叉比對 | |
| 5 | 人工審核：套用 `manual_review.csv` | |
| 6 | 合併同一間餐廳、從說明欄補營業時間與地址、TDX 轉座標 → `restaurants.json` | TDX |

**金鑰**：都放在 repo 根目錄 `.env`（`GEMINI_API_KEY=...`，[Google AI Studio](https://aistudio.google.com/apikey) 申請；TDX 金鑰原本就在）。模型名稱在 `config.toml` 的 `[llm]`。

**輸出全部在 `raw/` 與 `restaurants.json`，不進 git**（含逐字稿原文，D017）。

和台北版的差異：

- 抽的不是「景點＋旅客經驗」，而是 §9.2 的〈餐廳名稱、面向、情境標籤、情感、原文引文〉；標籤用 `Literal` 鎖在 §9.1 標籤表內。
- 引文查核沒過的片段**直接丟棄、不重試**（§9.2-2）；負向片段保留為反證（§9.2-3）。
- 店名查核加上**影片標題與說明欄**交叉比對（§9.2-6）。
- 沒有 PTT 與官方景點主檔；營業時間與地址改從說明欄取（`places.py`）。


## 步驟 1：設定

In [ ]:
import hashlib
import json
import re
import sys
from collections import Counter
from pathlib import Path

import pandas as pd

HERE = Path.cwd()                                   # 在 corpus/ 開這本 notebook
sys.path[:0] = [str(HERE.parent), str(HERE)]
from settings import secret, setting                # noqa: E402
import places                                       # noqa: E402  說明欄解析、TDX 地址轉座標

INPUT_JSON = HERE / "raw/youtube/youtube_taichung_food_cleaned.json"
if not INPUT_JSON.exists():
    raise FileNotFoundError(f"找不到 {INPUT_JSON}：請先執行 1_youtube爬蟲.ipynb 的步驟 6")
OUT_DIR = HERE / "raw/extract"
OUT_DIR.mkdir(parents=True, exist_ok=True)
LLM_DIR = OUT_DIR / "llm_output"
LLM_DIR.mkdir(exist_ok=True)
RESTAURANTS_JSON = HERE / "restaurants.json"       # 給 prototype/proto.py --data 用

## 步驟 2：清理字幕

1. 依換行與句尾標點切句，移除 Whisper 切壞的「�」。
2. 去重複：連續重複只留一句；6 字以上的句子若在前 8 句內出現過也刪掉（Whisper 會整段重播）。
3. 中文字比例低於 50% 的標記 `is_english=True`，抽取時略過，回爬蟲步驟 5 用 Whisper 重轉。

In [ ]:
SENTENCE_SPLIT = re.compile(r"[\n﹐，。！？!?；;]+")
WINDOW = 8
MIN_WINDOW_LEN = 6


def dedupe_sentences(text):
    """回傳 (清理後文字, 原句數, 刪掉的句數)"""
    text = (text or "").replace("\ufffd", "")
    sentences = [s.strip() for s in SENTENCE_SPLIT.split(text) if s.strip()]
    kept = []
    for s in sentences:
        if kept and s == kept[-1]:
            continue
        if len(s) >= MIN_WINDOW_LEN and s in kept[-WINDOW:]:
            continue
        kept.append(s)
    return "，".join(kept), len(sentences), len(sentences) - len(kept)


def chinese_ratio(text):
    cjk = len(re.findall(r"[\u4e00-\u9fff]", text))
    latin = len(re.findall(r"[A-Za-z]", text))
    return cjk / (cjk + latin) if cjk + latin else 0.0


videos = json.loads(INPUT_JSON.read_text(encoding="utf-8"))
cleaned = []
for v in videos:
    text, n_sentences, n_removed = dedupe_sentences(v["字幕內容"])
    cleaned.append({
        "video_id": v["影片ID"], "title": v["影片標題"], "channel": v["頻道"],
        "upload_date": v["上傳日期"], "fetched_at": v.get("取得時間", ""), "url": v["影片網址"],
        "transcript_source": v["字幕來源"], "transcript_lang": v["字幕語言"],
        "is_english": chinese_ratio(v["字幕內容"]) < 0.5,
        "chars_after": len(text), "removed_repeats": n_removed,
        "description": v["影片說明"], "description_raw": v.get("影片說明(原始)", v["影片說明"]),
        "transcript": text,
    })
clean_df = pd.DataFrame(cleaned)
print(f"影片 {len(clean_df)} 支，刪掉重複句 {clean_df['removed_repeats'].sum()} 句")
print(clean_df.groupby("transcript_source").size().to_string())
skip = clean_df[clean_df["is_english"] | (clean_df["chars_after"] == 0)]
if len(skip):
    print(f"\n[注意] 沒有中文字幕的 {len(skip)} 支會略過（回爬蟲步驟 5 用 Whisper 轉）：")
    print(skip[["video_id", "transcript_source", "title"]].to_string(index=False))

## 步驟 3：Gemini 抽取餐廳＋情境片段＋價格

**輸出格式**（Pydantic 強制 JSON，標籤鎖在 §9.1 標籤表）：

- `restaurants`：影片介紹的具名餐廳。`name` 校正後名稱、`name_raw` 字幕原文寫法、`branch` 分店。
- `snippets`：〈餐廳、面向、場合、同行對象、情感、引文〉。一則片段只屬於一間餐廳（§9.2-4）；負向也要抽（§9.2-3）。
- `prices`：字幕提到的價格。`low`／`high` 為每人金額；只提到單品價就填單品並 `per_person=false`。

**規則**：引文逐字複製（錯字照抄）；標籤只能依原文判斷，不靠常識推測（例如不可因為是牛排館就標「約會」）；字幕只有模糊說法（「這家」）而無法確定是哪間時不抽。

**試跑**：`TRIAL_LIMIT` 支（人工字幕優先，同類依情境關鍵字多寡）。每支結果存成 `raw/extract/llm_output/<影片ID>.json`，重跑會跳過已完成的；要重抽就刪掉它的 JSON。

In [ ]:
%pip install -q google-genai pydantic

In [ ]:
import os
import random
import time
from datetime import datetime
from typing import List, Literal, Optional

from google import genai
from google.genai import types
from pydantic import BaseModel, Field

MODEL = setting("llm", "model")
PROMPT_VERSION = "taichung-restaurant-v1"
THINKING_BUDGET = 512
REQUEST_TIMEOUT_SEC = 180
TRIAL_LIMIT = 5              # None = 全部

CONTEXT_KEYWORDS = ["約會", "慶生", "生日", "聚餐", "家庭", "朋友", "情侶", "一個人", "包廂", "氣氛", "CP", "價格", "元", "預約", "商務"]

# DESIGN.md §9.1 情境標籤表（新增或修改須先新增決策，D015）
Occasion = Literal["約會", "慶生", "朋友聚餐", "家庭聚餐", "商務", "一個人"]
Companion = Literal["情侶", "家庭", "朋友", "獨自"]
Aspect = Literal["料理風味", "用餐氛圍", "CP 值", "酒水飲料", "服務品質", "地點交通"]
Sentiment = Literal["正向", "負向", "中性"]


class RestaurantItem(BaseModel):
    name: str = Field(description="校正後的店名（繁體中文），不含分店")
    name_raw: str = Field(description="字幕中的原文寫法，可能有錯字")
    branch: str = Field(default="", description="分店名，例如 台中模範店；沒有就空字串")
    aliases: List[str] = Field(default_factory=list, description="文中出現的其他稱呼")
    district: str = Field(default="", description="文中提到或說明欄寫明的行政區，例如 西區；不確定留空")
    confidence: float = Field(description="0 到 1，這是可定位單一店家的信心")
    evidence: str = Field(description="逐字複製提到這間店的字幕原文（錯字照抄），最多 80 字")


class SnippetItem(BaseModel):
    restaurant_name: str = Field(description="對應 restaurants 裡的 name")
    aspect: Aspect
    occasions: List[Occasion] = Field(default_factory=list, description="原文明確支持或否定的場合；沒有就空陣列")
    companions: List[Companion] = Field(default_factory=list, description="原文明確提到的同行對象；沒有就空陣列")
    sentiment: Sentiment = Field(description="這段原文對該面向／情境的評價；負向如「太吵不適合約會」")
    quote: str = Field(description="逐字複製字幕原文（錯字照抄；多段用「…」連接，每段都要逐字），最多 100 字")


class PriceItem(BaseModel):
    restaurant_name: str
    low: Optional[int] = Field(default=None, description="金額下界（元）")
    high: Optional[int] = Field(default=None, description="金額上界（元）；只有一個數字時 low=high")
    per_person: bool = Field(description="是每人花費為 true；只是單品價為 false")
    evidence: str = Field(description="逐字複製提到價格的字幕原文，最多 60 字")


class ExtractionResponse(BaseModel):
    restaurants: List[RestaurantItem]
    snippets: List[SnippetItem]
    prices: List[PriceItem]


SYSTEM_INSTRUCTION = """你是餐廳情境知識抽取系統。輸入是一支「臺中美食 YouTube 影片」的標題、說明欄與字幕，從中抽取：
(1) 影片介紹的具名餐廳（restaurants）；(2) 對選餐廳有用的情境片段（snippets）；(3) 價格（prices）。

【字幕特性】
- 口語、無標點，可能是 Whisper 語音辨識，常有同音錯字（例如「台中6元」實為「台中肉員」）。
- 說明欄常列出正確店名，請用說明欄校正 name；name_raw 保留字幕原文寫法。無法確定就照原文並降低 confidence。

【restaurants】
- 只抽影片實際介紹、可定位的具名店家（餐廳、小吃、甜點、飲料、咖啡廳）。
- 不抽：泛稱（夜市、早餐店）、商品或食物名、只是順口提到的品牌、業配贊助的 App。
- 字幕只有模糊說法（「這家」「那間」）而無法確定是哪一家時，不要猜，寧可不抽。

【snippets：〈餐廳、面向、情境標籤、情感、原文引文〉】
- 面向 aspect 只能是：料理風味、用餐氛圍、CP 值、酒水飲料、服務品質、地點交通。
- 場合 occasions 只能是：約會、慶生、朋友聚餐、家庭聚餐、商務、一個人。
- 同行對象 companions 只能是：情侶、家庭、朋友、獨自。家庭涵蓋家人、親子、帶小孩與帶長輩；情侶涵蓋伴侶、約會與紀念日；朋友涵蓋三五好友、同事與團體。
- 標籤只能依原文明確線索判斷，不可依常識推測（例如不可因為是火鍋店就標「朋友聚餐」）；沒有線索就空陣列，只填 aspect。
- 負向評價也要抽（例如「人很多很吵」→ 用餐氛圍、負向），作為反證。
- 一則片段只屬於一間餐廳；同一段話在講別家時，不要掛到這家；無法判斷屬於哪家就不抽。
- 單純「好吃」也可以抽成 料理風味 正向，但要有完整原文。

【prices】
- 只抽字幕或說明欄明確說出的金額，不可換算或推測。

【引文寫法】
- quote／evidence 逐字複製原文，包含錯字（校正寫在 name，不要改引文）。多段用「…」連接，每段都必須是原文裡連續、逐字的文字。

【長度限制】quote 最多 100 字；evidence 最多 80 字；全部使用繁體中文。
"""

RETRYABLE = ("429", "500", "503", "504", "UNAVAILABLE", "RESOURCE_EXHAUSTED", "DEADLINE_EXCEEDED", "Timeout", "timed out")


class DailyQuotaExceeded(Exception):
    """免費方案「每天」的請求次數用完：重試沒用，要等隔天重置。"""


def generate(client, prompt, schema, max_retries=5):
    """呼叫 Gemini 並驗證 JSON 格式；伺服器忙時重試，每日額度用完或其他錯誤直接丟出"""
    config = types.GenerateContentConfig(
        system_instruction=SYSTEM_INSTRUCTION,
        thinking_config=types.ThinkingConfig(thinking_budget=THINKING_BUDGET),
        response_mime_type="application/json",
        response_schema=schema,
    )
    for attempt in range(max_retries + 1):
        try:
            resp = client.models.generate_content(model=MODEL, contents=prompt, config=config)
            return schema.model_validate_json(resp.text), resp.usage_metadata
        except Exception as e:
            message = f"{type(e).__name__}: {e}"
            if "PerDay" in message:
                raise DailyQuotaExceeded(message) from e
            if attempt == max_retries or not any(code in message for code in RETRYABLE):
                raise
            wait = min(60, 5 * 2 ** attempt) * (0.7 + random.random() * 0.6)
            print(f"    [{MODEL} 忙碌 {attempt + 1}/{max_retries}] {wait:.0f} 秒後重試")
            time.sleep(wait)


def doc_text(v):
    return (f"標題：{v['title']}\n頻道：{v['channel']}\n發布日期：{v['upload_date']}\n\n"
            f"【說明欄】\n{v['description_raw'][:3000]}\n\n【字幕】\n{v['transcript']}")


def save_result(v, parsed, usage):
    (LLM_DIR / f"{v['video_id']}.json").write_text(json.dumps({
        "video_id": v["video_id"], "model": MODEL, "prompt_version": PROMPT_VERSION,
        "extracted_at": datetime.now().isoformat(timespec="seconds"),
        "usage": {"total": getattr(usage, "total_token_count", None)},
        **parsed.model_dump(),
    }, ensure_ascii=False, indent=1), encoding="utf-8")


client = genai.Client(api_key=secret("GEMINI_API_KEY"),
                      http_options=types.HttpOptions(timeout=REQUEST_TIMEOUT_SEC * 1000))
SOURCE_RANK = {"人工字幕": 0, "自動字幕": 1}
candidates = sorted((v for v in cleaned if not v["is_english"] and v["chars_after"]),
                    key=lambda v: (SOURCE_RANK.get(v["transcript_source"], 2),
                                   -sum(v["transcript"].count(k) for k in CONTEXT_KEYWORDS)))
targets = candidates[:TRIAL_LIMIT] if TRIAL_LIMIT else candidates

done, failed = 0, []
for i, v in enumerate(targets, 1):
    if (LLM_DIR / f"{v['video_id']}.json").exists():
        print(f"[{i}/{len(targets)}] 已完成，略過：{v['title'][:40]}")
        continue
    print(f"[{i}/{len(targets)}] {v['title'][:40]}（{v['chars_after']:,} 字）")
    try:
        parsed, usage = generate(client, doc_text(v), ExtractionResponse)
    except DailyQuotaExceeded:
        print(f"    [停止] {MODEL} 今天的免費額度用完，明天重跑這格")
        failed += [t["video_id"] for t in targets[i - 1:]]
        break
    except Exception as e:
        print(f"    [失敗] {type(e).__name__}: {e}")
        failed.append(v["video_id"])
        continue
    save_result(v, parsed, usage)
    done += 1
    print(f"    餐廳 {len(parsed.restaurants)}、片段 {len(parsed.snippets)}、價格 {len(parsed.prices)}")
    time.sleep(0.5)

print(f"\n--- 完成 --- 新抽取 {done} 支、失敗 {len(failed)} 支")
if failed:
    print("失敗的（重跑這格會再試）：", failed)

## 步驟 4：攤平並自動查核（不用 LLM）

| 查核 | 規則 | 沒通過 |
|---|---|---|
| 片段 `quote_verified` | 引文用「…」切段，每段（4 字以上）都能在字幕或說明欄逐字找到（忽略標點空白） | **直接丟棄**（§9.2-2） |
| 片段歸屬 | `restaurant_name` 要是同一支影片抽出的餐廳 | 丟棄（§9.2-4） |
| 價格 `verified` | evidence 逐字找到，且 low／high 的數字出現在 evidence（中文數字先轉阿拉伯數字） | 不採用 |
| 店名 `name_check` | `verified`＝店名出現在字幕、標題或說明欄；`expanded`＝店名包含字幕原文；`review`＝都不是，要人工看（§9.2-6） | 進人工審核 |


In [ ]:
PUNCT = re.compile(r"[\s，,。．.、！!？?；;：:「」『』（）()\"'…~～\-—]+")


def squash(text):
    return PUNCT.sub("", str(text or ""))


def evidence_found(evidence, source):
    segments = [squash(seg) for seg in re.split(r"…|\.\.\.", str(evidence or ""))]
    segments = [seg for seg in segments if len(seg) >= 4]
    return bool(segments) and all(seg in source for seg in segments)


CN_DIGITS = {"〇": 0, "零": 0, "一": 1, "二": 2, "兩": 2, "三": 3, "四": 4, "五": 5, "六": 6, "七": 7, "八": 8, "九": 9}
CN_NUMBER = re.compile(r"[〇零一二兩三四五六七八九十百千]+")


def cn_to_int(text):
    """九→9、十五→15、一百五→150、兩千→2000"""
    if all(ch in CN_DIGITS for ch in text):
        return int("".join(str(CN_DIGITS[ch]) for ch in text))
    total, current = 0, 0
    for ch in text:
        if ch in CN_DIGITS:
            current = CN_DIGITS[ch]
        elif ch in "十百千":
            total += (current or 1) * {"十": 10, "百": 100, "千": 1000}[ch]
            current = 0
    if len(text) >= 2 and text[-2] in "百千" and text[-1] in CN_DIGITS:   # 一百五 → 150、兩千五 → 2500
        current *= {"百": 10, "千": 100}[text[-2]]
    return total + current


def to_digits(text):
    text = str(text or "").translate(str.maketrans("０１２３４５６７８９", "0123456789"))
    return CN_NUMBER.sub(lambda m: str(cn_to_int(m.group())), text)


assert cn_to_int("九") == 9 and cn_to_int("十五") == 15 and cn_to_int("一百五") == 150 and cn_to_int("兩千") == 2000
assert evidence_found("這家湯包…其實很難吃", squash("這家湯包真的很有名")) is False   # 每段（4 字以上）都要逐字找到
assert evidence_found("這家湯包真的…真的很有名", squash("這家湯包真的很有名"))

meta = {v["video_id"]: v for v in cleaned}
source_text = {vid: squash(v["transcript"] + v["description_raw"]) for vid, v in meta.items()}
cross_text = {vid: squash(v["transcript"] + v["title"] + v["description_raw"]) for vid, v in meta.items()}

rest_rows, snip_rows, price_rows = [], [], []
for path in sorted(LLM_DIR.glob("*.json")):
    r = json.loads(path.read_text(encoding="utf-8"))
    vid = r["video_id"]
    src = {"video_id": vid, "video_url": meta.get(vid, {}).get("url"), "upload_date": meta.get(vid, {}).get("upload_date"),
           "transcript_source": meta.get(vid, {}).get("transcript_source"), "model": r["model"], "prompt_version": r["prompt_version"]}
    rest_rows += [{**x, **src} for x in r["restaurants"]]
    snip_rows += [{**x, **src} for x in r["snippets"]]
    price_rows += [{**x, **src} for x in r["prices"]]

rests = pd.DataFrame(rest_rows)
snips = pd.DataFrame(snip_rows)
prices = pd.DataFrame(price_rows)
if rests.empty:
    raise SystemExit("還沒有抽取結果：先跑步驟 3")


def name_check(name, raw, vid):
    if squash(name) in cross_text.get(vid, ""):
        return "verified"
    raw = squash(raw)
    return "expanded" if len(raw) >= 2 and raw in squash(name) else "review"


rests["name_check"] = [name_check(n, raw, v) for n, raw, v in zip(rests["name"], rests["name_raw"], rests["video_id"])]
rests["evidence_verified"] = [evidence_found(e, source_text.get(v, "")) for e, v in zip(rests["evidence"], rests["video_id"])]

known = set(zip(rests["video_id"], rests["name"]))
if len(snips):
    snips["quote_verified"] = [evidence_found(q, source_text.get(v, "")) for q, v in zip(snips["quote"], snips["video_id"])]
    snips["owner_found"] = [(v, n) in known for v, n in zip(snips["video_id"], snips["restaurant_name"])]
if len(prices):
    def price_ok(row):
        ev = to_digits(re.sub(r"\s+", "", str(row["evidence"])))
        nums = [str(int(x)) for x in (row["low"], row["high"]) if pd.notna(x)]
        return bool(nums) and evidence_found(row["evidence"], source_text.get(row["video_id"], "")) and all(n in ev for n in nums)
    prices["verified"] = prices.apply(price_ok, axis=1)

rests.to_csv(OUT_DIR / "restaurants_extracted.csv", index=False, encoding="utf-8-sig")
snips.to_csv(OUT_DIR / "snippets_extracted.csv", index=False, encoding="utf-8-sig")
prices.to_csv(OUT_DIR / "prices_extracted.csv", index=False, encoding="utf-8-sig")

print(f"餐廳 {len(rests)} 筆，店名查核：{rests['name_check'].value_counts().to_dict()}")
if len(snips):
    ok = snips["quote_verified"] & snips["owner_found"]
    print(f"片段 {len(snips)} 則，通過 {ok.sum()}（引文找不到 {(~snips['quote_verified']).sum()}、歸屬不到餐廳 {(~snips['owner_found']).sum()}）→ 沒通過的直接丟棄")
    print("情感：", snips.loc[ok, "sentiment"].value_counts().to_dict())
    print("場合：", dict(Counter(t for s in snips.loc[ok, "occasions"] for t in s)))
if len(prices):
    print(f"價格 {len(prices)} 筆，通過 {prices['verified'].sum()}")
guessed = rests[rests["name_check"] == "review"]
if len(guessed):
    print("\n[請人工確認] 店名不在字幕、標題、說明欄裡（可能是校正，也可能是猜的）：")
    print("  " + "、".join(f"{r['name_raw']}→{r['name']}" for _, r in guessed.iterrows()))

## 步驟 5：人工審核

步驟 4 標成 `review` 的店名記錄在 `raw/extract/manual_review.csv`。用 Excel 或 VS Code 打開，填 `decision`：

- `keep`：保留
- `rename`：改名，新名稱填 `corrected_name`
- `drop`：刪除（底下的片段與價格一起刪）

存檔後重跑這格。新的待確認項目會加到表的最後，已填的決定不會被改掉。

In [ ]:
REVIEW_CSV = OUT_DIR / "manual_review.csv"
REVIEW_COLUMNS = ["video_id", "name", "name_raw", "evidence", "decision", "corrected_name", "note"]

review = pd.read_csv(REVIEW_CSV, dtype=str).fillna("") if REVIEW_CSV.exists() else pd.DataFrame(columns=REVIEW_COLUMNS)
seen = set(zip(review["video_id"], review["name"]))
new_rows = [{c: r.get(c, "") for c in REVIEW_COLUMNS} for _, r in rests[rests["name_check"] == "review"].iterrows()
            if (r["video_id"], r["name"]) not in seen]
if new_rows:
    review = pd.concat([review, pd.DataFrame(new_rows, columns=REVIEW_COLUMNS)], ignore_index=True)
review = review.fillna("")
review.to_csv(REVIEW_CSV, index=False, encoding="utf-8-sig")

decision = {(r["video_id"], r["name"]): (r["decision"].strip().lower(), r["corrected_name"].strip()) for _, r in review.iterrows()}
final = rests.copy()
final["decision"] = [decision.get((v, n), ("", ""))[0] for v, n in zip(final["video_id"], final["name"])]
final = final[(final["decision"] != "drop") & ((final["name_check"] != "review") | final["decision"].isin(["keep", "rename"]))]
renamed = {k: new for k, (d, new) in decision.items() if d == "rename" and new}
final["final_name"] = [renamed.get((v, n), n) for v, n in zip(final["video_id"], final["name"])]

pending = review[review["decision"].str.strip() == ""]
print(f"審核表 {len(review)} 項（待填 {len(pending)}）→ {REVIEW_CSV}")
print(f"可用餐廳 {len(final)}／{len(rests)}")

## 步驟 6：合併同一間餐廳 → `restaurants.json`

- **同一間**：正規化店名（去空白與符號、臺→台）＋分店相同。ponytail：只靠字面比對；§9.2-5 的「向量找候選＋LLM 判定」之後再做。
- **營業時間、地址**：從影片原始說明欄解析（`places.parse_description`），店名互相包含就算對上；觀測時間記為影片發布日。
- **座標**：用說明欄地址呼叫 TDX 地址轉座標（快取在 `raw/geocode.json`，每查一筆等 13 秒）。**沒有地址的店拿不到座標，先排除**，之後改用 Places（§9.2-5）。
- **價格**：通過查核且 `per_person=true` 的，取所有金額的最小值到最大值當區間；沒有就留空（預算判 U，O012）。
- **片段**：正向／中性的放 `reviews`（附標籤，給原型的情境召回）；負向的放 `counter`（反證，§9.2-3）。

In [ ]:
def normalize(name):
    return re.sub(r"[\W_]+", "", str(name)).replace("臺", "台")


ok_snips = snips[snips["quote_verified"] & snips["owner_found"]] if len(snips) else snips
ok_prices = prices[prices["verified"] & prices["per_person"]] if len(prices) else prices
desc_shops = {vid: places.parse_description(v["description_raw"]) for vid, v in meta.items()}


def from_description(vid, name):
    key = normalize(name)
    for s in desc_shops.get(vid, []):
        k = normalize(s["name"])
        if k and (k in key or key in k):
            return s
    return None


def tagged(row):
    tags = "、".join(list(row["occasions"]) + list(row["companions"]))
    return f"【{row['aspect']}{'｜' + tags if tags else ''}】{row['quote']}"


out, no_geo = [], []
final["entity"] = [f"{normalize(n)}|{normalize(b)}" for n, b in zip(final["final_name"], final["branch"])]
for key, g in final.groupby("entity", sort=False):
    name = Counter(g["final_name"]).most_common(1)[0][0]
    pairs = set(zip(g["video_id"], g["name"]))
    mine = ok_snips[[(v, n) in pairs for v, n in zip(ok_snips["video_id"], ok_snips["restaurant_name"])]] if len(ok_snips) else ok_snips
    my_prices = ok_prices[[(v, n) in pairs for v, n in zip(ok_prices["video_id"], ok_prices["restaurant_name"])]] if len(ok_prices) else ok_prices
    shop = next((s for v, n in pairs if (s := from_description(v, n))), None)
    latlng = places.geocode(shop["address"], HERE / "raw/geocode.json") if shop else None
    if not latlng:
        no_geo.append(name)
        continue
    nums = [int(x) for col in ("low", "high") for x in my_prices.get(col, []) if pd.notna(x)]
    out.append({
        "id": "ext-" + hashlib.sha1(key.encode()).hexdigest()[:10],
        "name": name + (f"（{g['branch'].iloc[0]}）" if g["branch"].iloc[0] else ""),
        "aliases": sorted(set(g["name"]) | set(g["name_raw"]) - {name}),
        "address": shop["address"], "lat": latlng[0], "lng": latlng[1],
        "price": [min(nums), max(nums)] if nums else None,
        "hours": shop["hours"], "closed_weekdays": shop["closed_weekdays"],
        "reviews": [tagged(r) for _, r in mine[mine["sentiment"] != "負向"].iterrows()] if len(mine) else [],
        "counter": [tagged(r) for _, r in mine[mine["sentiment"] == "負向"].iterrows()] if len(mine) else [],
        "sources": [{"video": meta[v]["url"], "published_at": meta[v]["upload_date"], "fetched_at": meta[v]["fetched_at"],
                     "transcript_source": meta[v]["transcript_source"]} for v in dict.fromkeys(g["video_id"])],
        "hours_from": "影片說明欄（觀測時間＝影片發布日）",
    })

RESTAURANTS_JSON.write_text(json.dumps({"_note": "由 corpus/2_餐廳抽取.ipynb 產生，含逐字稿原文，不進 git（D017）。",
                                        "restaurants": out}, ensure_ascii=False, indent=1), encoding="utf-8")
print(f"→ {RESTAURANTS_JSON}：{len(out)} 家（有價格 {sum(1 for r in out if r['price'])}、有情境片段 {sum(1 for r in out if r['reviews'])}）")
if no_geo:
    print(f"[略過] 說明欄沒有地址、拿不到座標的 {len(no_geo)} 家：{'、'.join(no_geo)}")
print("\n跑原型：cd ../prototype && python proto.py --data ../corpus/restaurants.json --context 慶生 --budget 600 --depart 2026-11-20T19:00 --origin 24.1372,120.6869")